In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as  sns

In [2]:
df1=pd.read_csv("Bengaluru_House_Data.csv")

In [3]:
df1.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [4]:
df2=df1.drop(["area_type","society","balcony","availability"],axis=1)
df2.shape

(13320, 5)

In [5]:
df2.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

In [6]:
df3=df2.dropna()
df3.isnull().sum()

location      0
size          0
total_sqft    0
bath          0
price         0
dtype: int64

In [7]:
df3.shape

(13246, 5)

In [8]:
df3["bhk"]=df3["size"].apply(lambda x: int(x.split()[0]))
df3.bhk.unique()

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_26408\2735452439.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3["bhk"]=df3["size"].apply(lambda x: int(x.split()[0]))


array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18], dtype=int64)

In [9]:
df3.head()

,location,size,total_sqft,bath,price,bhk
0,Electronic City Phase II,2 BHK,1056,2.0,39.07,2
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00,4
2,Uttarahalli,3 BHK,1440,2.0,62.00,3
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00,3
4,Kothanur,2 BHK,1200,2.0,51.00,2


In [10]:
df3["total_sqft"].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

In [11]:
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

In [12]:
df3[~df3["total_sqft"].apply(is_float)].head(10)

,location,size,total_sqft,bath,price,bhk
30,Yelahanka,4 BHK,2100 - 2850,4.0,186.000,4
122,Hebbal,4 BHK,3067 - 8156,4.0,477.000,4
137,8th Phase JP Nagar,2 BHK,1042 - 1105,2.0,54.005,2
165,Sarjapur,2 BHK,1145 - 1340,2.0,43.490,2
188,KR Puram,2 BHK,1015 - 1540,2.0,56.800,2
410,Kengeri,1 BHK,34.46Sq. Meter,1.0,18.500,1
549,Hennur Road,2 BHK,1195 - 1440,2.0,63.770,2
648,Arekere,9 Bedroom,4125Perch,9.0,265.000,9
661,Yelahanka,2 BHK,1120 - 1145,2.0,48.130,2
672,Bettahalsoor,4 Bedroom,3090 - 5002,4.0,445.000,4


In [13]:
s1="3090-5002"
x=s1.split("-")
(int(x[0])+int(x[1]))/2

4046.0

In [14]:
# Above data shows that total sqft can be a range (E.G 2100-2800). For such cases we can just take average of min & max value in the range.
# There are other cases where values are in SQM which can be converted to sqft using unit conversion.

In [15]:
def convert_sqft_to_num(x):
    tokens= x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0])+float(tokens[1]))/2
    try:
        return float(x)
    except:
        return None
            

In [16]:
df3.shape

(13246, 6)

In [17]:
df4=df3.copy()
df4.total_sqft=df4.total_sqft.apply(convert_sqft_to_num)
df4=df4[df4.total_sqft.notnull()]
df4.shape

(13200, 6)

In [18]:
# Addd new feature called price per square feet
df5=df4.copy()
df5["price_per_sqft"] = df5["price"]*100000/df5["total_sqft"]
df5.head()

,location,size,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2,4250.000000


In [19]:
# Examine locations which is a categorical variable. we need to apply the dimensionality reduction technique here to reduce the number of 
# locations

df5.location =df5.location.apply(lambda x: x.strip())
location_stats= df5["location"].value_counts(ascending=False)
location_stats

location
Whitefield                   533
Sarjapur  Road               392
Electronic City              304
Kanakpura Road               264
Thanisandra                  235
                            ... 
Rajanna Layout                 1
Subramanyanagar                1
Lakshmipura Vidyaanyapura      1
Malur Hosur Road               1
Abshot Layout                  1
Name: count, Length: 1287, dtype: int64

In [20]:
len(location_stats[location_stats>10])

240

In [21]:
len(location_stats)

1287

In [22]:
len(location_stats[location_stats<=10])

1047

In [23]:
# Dimensionality Reductions
# Any location having less than 10 data pints should be tagged as "othetr" location. this way number of categories can be reduced by huge
# ammount. later on when we do one hot encoding, it will help us with having fewer dummy columns.


In [24]:
location_stats_less_than_10= location_stats[location_stats<=10]
location_stats_less_than_10

location
BTM 1st Stage                10
Gunjur Palya                 10
Nagappa Reddy Layout         10
Sector 1 HSR Layout          10
Thyagaraja Nagar             10
                             ..
Rajanna Layout                1
Subramanyanagar               1
Lakshmipura Vidyaanyapura     1
Malur Hosur Road              1
Abshot Layout                 1
Name: count, Length: 1047, dtype: int64

In [25]:
len(df5.location.unique())

1287

In [26]:
df5.location = df5.location.apply(lambda x: "Unknown" if x in location_stats_less_than_10  else x)
len(df5.location.unique())

241

In [27]:
df5.location.value_counts()

location
Unknown            2872
Whitefield          533
Sarjapur  Road      392
Electronic City     304
Kanakpura Road      264
                   ... 
Doddaballapur        11
Tindlu               11
Marsur               11
HAL 2nd Stage        11
Kodigehalli          11
Name: count, Length: 241, dtype: int64

In [28]:
df5["bhk"].unique()

array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18], dtype=int64)

In [29]:
df5["bhk"]=df5.bhk.apply(lambda x: x if x<=11 else np.nan)
df5.bhk.unique()

array([ 2.,  4.,  3.,  6.,  1.,  8.,  7.,  5., 11.,  9., nan, 10.])

In [30]:
df5.isnull().sum()

location          0
size              0
total_sqft        0
bath              0
price             0
bhk               8
price_per_sqft    0
dtype: int64

In [31]:
df5["bhk"].fillna(df5["bhk"].mode()[0],inplace=True)

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_26408\1135971696.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df5["bhk"].fillna(df5["bhk"].mode()[0],inplace=True)


In [32]:
df5.head(10)

,location,size,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2.0,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4.0,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3.0,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3.0,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2.0,4250.000000
5,Whitefield,2 BHK,1170.0,2.0,38.00,2.0,3247.863248
6,Old Airport Road,4 BHK,2732.0,4.0,204.00,4.0,7467.057101
7,Rajaji Nagar,4 BHK,3300.0,4.0,600.00,4.0,18181.818182
8,Marathahalli,3 BHK,1310.0,3.0,63.25,3.0,4828.244275
9,Unknown,6 Bedroom,1020.0,6.0,370.00,6.0,36274.509804


In [33]:
# Outlier Removal using business logic
# As a data scientist is a data scientist when you have a conversation with your business manager (who has experience in real estate).
# He will tell you that normally square ft per bedroom is 300(i.e 2 bhk appartment is minimum 600 sqft. 
# if you have example 400 sqft appartment with 2 bhk than that seems suspicious and can be removed as an outlier.
# WE will removes such outliers by keeping our minimum there sold per bhk to be 300 sqft.

In [34]:
df5[df5.total_sqft/df5.bhk<300].head()

,location,size,total_sqft,bath,price,bhk,price_per_sqft
9,Unknown,6 Bedroom,1020.0,6.0,370.0,6.0,36274.509804
45,HSR Layout,8 Bedroom,600.0,9.0,200.0,8.0,33333.333333
58,Murugeshpalya,6 Bedroom,1407.0,4.0,150.0,6.0,10660.980810
68,Devarachikkanahalli,8 Bedroom,1350.0,7.0,85.0,8.0,6296.296296
70,Unknown,3 Bedroom,500.0,3.0,100.0,3.0,20000.000000


In [70]:
df5.shape

(13200, 7)

In [72]:
df6=df5[~(df5.total_sqft/df5.bhk<300)]
df6.shape

(12462, 7)

In [76]:
df6.bath.unique()

array([ 2.,  5.,  3.,  4.,  1.,  8.,  6.,  7.,  9., 14., 27., 12., 16.,
       40., 15., 10., 13., 18.])

In [78]:
df6[df6.bath>10]

,location,size,total_sqft,bath,price,bhk,price_per_sqft
1078,Unknown,9 Bedroom,3300.0,14.0,500.0,9.0,15151.515152
1718,Unknown,27 BHK,8000.0,27.0,230.0,2.0,2875.000000
3096,Unknown,10 BHK,12000.0,12.0,525.0,10.0,4375.000000
3379,Unknown,19 BHK,2000.0,16.0,490.0,2.0,24500.000000
3609,Unknown,16 BHK,10000.0,16.0,550.0,2.0,5500.000000
4684,Munnekollal,43 Bedroom,2400.0,40.0,660.0,2.0,27500.000000
4916,Unknown,14 BHK,1250.0,15.0,125.0,2.0,10000.000000
7979,Unknown,11 BHK,6000.0,12.0,150.0,11.0,2500.000000
8636,Neeladri Nagar,10 BHK,4000.0,12.0,160.0,10.0,4000.000000
9935,Unknown,13 BHK,5425.0,13.0,275.0,2.0,5069.124424


In [82]:
# It is unusal to have 2 more bathrooms than number of bedroom in a home
df6[df6.bath>df6.bhk+2]

,location,size,total_sqft,bath,price,bhk,price_per_sqft
1078,Unknown,9 Bedroom,3300.0,14.0,500.0,9.0,15151.515152
1718,Unknown,27 BHK,8000.0,27.0,230.0,2.0,2875.000000
2620,Unknown,6 BHK,11338.0,9.0,1000.0,6.0,8819.897689
3379,Unknown,19 BHK,2000.0,16.0,490.0,2.0,24500.000000
3609,Unknown,16 BHK,10000.0,16.0,550.0,2.0,5500.000000
4684,Munnekollal,43 Bedroom,2400.0,40.0,660.0,2.0,27500.000000
4916,Unknown,14 BHK,1250.0,15.0,125.0,2.0,10000.000000
6533,Mysore Road,12 Bedroom,2232.0,6.0,300.0,2.0,13440.860215
6838,Rajaji Nagar,5 BHK,7500.0,8.0,1700.0,5.0,22666.666667
7709,Chikkabanavar,4 Bedroom,2460.0,7.0,80.0,4.0,3252.032520


In [84]:
# Again the business mananger has a conversation with you(i.e Data sceintist) thst if you have a 4 bedroom home and 
#Even if you have a bathroom in all 4 rooms plus one geust bathroom,you will have a total bath= total bed+max + 1.
# anything above that is an outlier or a data eroor and can be removed

In [86]:
df7= df6[df6.bath<df6.bhk+2]
df7.shape

(12301, 7)

In [88]:
df8=df7.drop(["size","price_per_sqft"],axis="columns")
df8.head(3)

,location,total_sqft,bath,price,bhk
0,Electronic City Phase II,1056.0,2.0,39.07,2.0
1,Chikka Tirupathi,2600.0,5.0,120.00,4.0
2,Uttarahalli,1440.0,2.0,62.00,3.0


In [96]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df8["location"]=le.fit_transform(df8["location"])


In [98]:
df8.head()

,location,total_sqft,bath,price,bhk
0,79,1056.0,2.0,39.07,2.0
1,60,2600.0,5.0,120.00,4.0
2,226,1440.0,2.0,62.00,3.0
3,159,1521.0,3.0,95.00,3.0
4,151,1200.0,2.0,51.00,2.0


In [104]:
X=df8.drop("price",axis=1)
y=df8["price"]

In [106]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_sc=sc.fit_transform(X)

In [110]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_sc,y,test_size=0.2,random_state=42)

In [114]:
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr.fit(X_train,y_train)

LinearRegression()

In [116]:
y_pred=lr.predict(X_test)

In [120]:
calc=pd.DataFrame(np.c_[y_test,y_pred],columns=["Original Price","Predicted Price"])

In [122]:
calc

,Original Price,Predicted Price
0,18.0,1.725071
1,23.5,-4.585549
2,128.0,58.723748
3,45.0,70.873234
4,75.0,58.692959
...,...,...
2456,95.0,70.870038
2457,155.0,158.846305
2458,72.0,54.716788
2459,115.0,91.254102
